# Calculate affected geohash overlap percentages

For NASA flood IDs 4775 and 4859, this notebook calculates the percentage of each affected geohash-5 cell that overlaps the NASA flood polygon (`dis_geo`). Affected geohashes are the unique `geohash_5` values with `area_affected == 1` in each current `disaster_<flood_id>_agg_info.csv`. Outputs are sorted by overlap percentage increasingly.

In [ ]:
import os

import geopandas as gpd
import pandas as pd

import utils

FLOOD_IDS = [4775, 4859]
FLOOD_ARCHIVE_PATH = "../data/FloodArchive_NASA/FloodArchive_region.shp"
RESULT_ROOT = "../outputs"
EQUAL_AREA_CRS = "EPSG:6933"

AGG_INFO_PATHS = {
    flood_id: os.path.join("../data", f"disaster_{flood_id}_agg_info.csv")
    for flood_id in FLOOD_IDS
}

## Load NASA flood geometries

In [ ]:
flood_archive = gpd.read_file(FLOOD_ARCHIVE_PATH)

if flood_archive.crs is None:
    flood_archive = flood_archive.set_crs(epsg=4326)
else:
    flood_archive = flood_archive.to_crs(epsg=4326)

flood_archive = flood_archive[flood_archive["ID"].isin(FLOOD_IDS)].copy()
flood_archive[["ID", "COUNTRY", "BEGAN", "ENDED", "geometry"]]

## Calculate overlap percentages

In [ ]:
def read_affected_geohashes(flood_id):
    agg_path = AGG_INFO_PATHS[flood_id]
    agg_info = pd.read_csv(agg_path)
    return sorted(agg_info.loc[agg_info["area_affected"] == 1, "geohash_5"].dropna().unique())


def build_geohash_gdf(geohashes):
    geohash_gdf = pd.DataFrame({"geohash_5": geohashes})
    geohash_gdf["geometry"] = geohash_gdf["geohash_5"].apply(utils.geohash_to_polygon)
    return gpd.GeoDataFrame(geohash_gdf, geometry="geometry", crs="EPSG:4326")


def calculate_overlap_percentage(flood_id):
    affected_geohashes = read_affected_geohashes(flood_id)
    geohash_gdf = build_geohash_gdf(affected_geohashes).to_crs(EQUAL_AREA_CRS)

    dis_geo = flood_archive.loc[flood_archive["ID"] == flood_id, "geometry"].iloc[0]
    dis_geo = gpd.GeoSeries([dis_geo], crs="EPSG:4326").to_crs(EQUAL_AREA_CRS).iloc[0]

    geohash_gdf["geohash_area_m2"] = geohash_gdf.geometry.area
    geohash_gdf["overlap_area_m2"] = geohash_gdf.geometry.intersection(dis_geo).area
    geohash_gdf["overlap_percentage"] = (
        geohash_gdf["overlap_area_m2"] / geohash_gdf["geohash_area_m2"] * 100
    )

    overlap = geohash_gdf[
        ["geohash_5", "geohash_area_m2", "overlap_area_m2", "overlap_percentage"]
    ].copy()
    overlap.insert(0, "flood_id", flood_id)
    return overlap.sort_values(["overlap_percentage", "geohash_5"]).reset_index(drop=True)


outputs = []
for flood_id in FLOOD_IDS:
    overlap = calculate_overlap_percentage(flood_id)
    output_path = os.path.join(
        "../data",
        f"disaster_{flood_id}_affected_geohash_overlap_percentage.csv",
    )
    overlap.to_csv(output_path, index=False)
    outputs.append(
        {
            "flood_id": flood_id,
            "affected_geohashes": len(read_affected_geohashes(flood_id)),
            "output_rows": len(overlap),
            "count_matches": len(overlap) == len(read_affected_geohashes(flood_id)),
            "output_path": output_path,
        }
    )

pd.DataFrame(outputs)

In [ ]:
pd.concat([calculate_overlap_percentage(flood_id).head() for flood_id in FLOOD_IDS])